# PLayer Goal and Shot Creation

### Import Libraries

In [2]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import requests
from lxml import html
from bs4 import BeautifulSoup


### Create web pages variables to display player Goal and shot statistics by league

In [ ]:
pl = "https://fbref.com/en/comps/9/gca/Premier-League-Stats"
liga = "http://fbref.com/en/comps/12/gca/La-Liga-Stats"
seriea = "https://fbref.com/en/comps/11/gca/Serie-A-Stats"
bundesliga = "https://fbref.com/en/comps/20/gca/Bundesliga-Stats"
ligue1 = "https://fbref.com/en/comps/13/gca/Ligue-1-Stats"


### Create DataFrame

In [3]:
FBREFplayer = pd.DataFrame(columns = [
    'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', '90s',
    'SCA', 'SCA90', 
    'SCA_PassLive', 'SCA_PassDead', 'SCA_TO', 'SCA_Sh', 'SCA_Fld', 'SCA_Def',
    'GCA', 'GCA90',
    'GCA_PassLive', 'GCA_PassDead', 'GCA_TO', 'GCA_Sh', 'GCA_Fld', 'GCA_Def'
])

### Defining path variable

In [13]:
brave_path = "C:/Program Files/BraveSoftware/Brave-Browser/Application/brave.exe" 

### playerscraper Function
Scrapes Player statistics from a league webpage URL and inserts the extracted data into a pandas DataFrame

In [ ]:
# Create a list to hold infixed rows
infixed_rows = []
positions = ['GK', 'DF', 'MF', 'FW', 'FW,DF', 'FW,MF', 'MF,DF', 'MF,FW']

def playerscraper (url):
    # First l'ets set up the web driver with the Brave browser
    options = Options()
    options.binary_location = brave_path
    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get(url)

    # define a variable to hold the GKs table
    table = driver.find_element(By.XPATH , '//*[@id="stats_gca"]/tbody')

    # Get all rows in the table
    rows = table.find_elements(By.TAG_NAME, 'tr')

    # Loop through each row and extract the data
    for row in rows:
        # transform the row data into a list
        row_data = row.text.split(' ')
        
        # skip the header rows
        if row_data[0] == 'Rk':
            continue
        # Remove the first element (the rank)
        row_data.pop(0)  

        # Remove the 'Matches' element if it exists
        row_data.remove('Matches')
        
        # for players with name of one word
        if row_data[3] in positions:
            # remove the duplicate of nationality            
            row_data.pop(1)
            
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                # If the team name is split, combine it
                team = row_data.pop(4) + ' ' + row_data.pop(4)
                row_data.insert(4, team)

        # for players with name of more than one word
        elif row_data[4] in positions:
            # remove the duplicate of nationality
            row_data.pop(2)

            # Fixing the player name and team
            name = row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[4]) > 2 or row_data[4] == '05':
                # If the team name is split, combine it
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)

        # for players with name of 3 parts
        elif row_data[5] in positions:
            # remove the duplicate of nationality
            row_data.pop(3)
            # Fixing the player name and team
            name = row_data.pop(0) + ' ' + row_data.pop(0)+ ' '+ row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[6]) > 2 or row_data[6] == '05':
                # If the team name is split, combine it
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        else :
            # If the player name is not in the expected format, skip the row
            continue

        # Check if the row has the expected number of elements
        if len(row_data) != 23:
            continue
        # insert the row data into the DataFrame
        FBREFplayer.loc[len(FBREFplayer)] = row_data
    # Convert the DataFrame columns to appropriate data types
    FBREFplayer['90s'] = pd.to_numeric(FBREFplayer['90s'], errors='coerce')
    FBREFplayer['SCA'] = pd.to_numeric(FBREFplayer['SCA'], errors='coerce')
    FBREFplayer['SCA90'] = pd.to_numeric(FBREFplayer['SCA90'], errors='coerce')
    FBREFplayer['SCA_PassLive'] = pd.to_numeric(FBREFplayer['SCA_PassLive'], errors='coerce')
    FBREFplayer['SCA_PassDead'] = pd.to_numeric(FBREFplayer['SCA_PassDead'], errors='coerce')
    FBREFplayer['SCA_TO'] = pd.to_numeric(FBREFplayer['SCA_TO'], errors='coerce')
    FBREFplayer['SCA_Sh'] = pd.to_numeric(FBREFplayer['SCA_Sh'], errors='coerce')
    FBREFplayer['SCA_Fld'] = pd.to_numeric(FBREFplayer['SCA_Fld'], errors='coerce')
    FBREFplayer['SCA_Def'] = pd.to_numeric(FBREFplayer['SCA_Def'], errors='coerce')
    FBREFplayer['GCA'] = pd.to_numeric(FBREFplayer['GCA'], errors='coerce')
    FBREFplayer['GCA90'] = pd.to_numeric(FBREFplayer['GCA90'], errors='coerce')
    FBREFplayer['GCA_PassLive'] = pd.to_numeric(FBREFplayer['GCA_PassLive'], errors='coerce')
    FBREFplayer['GCA_PassDead'] = pd.to_numeric(FBREFplayer['GCA_PassDead'], errors='coerce')
    FBREFplayer['GCA_TO'] = pd.to_numeric(FBREFplayer['GCA_TO'], errors='coerce')
    FBREFplayer['GCA_Sh'] = pd.to_numeric(FBREFplayer['GCA_Sh'], errors='coerce')
    FBREFplayer['GCA_Fld'] = pd.to_numeric(FBREFplayer['GCA_Fld'], errors='coerce')
    FBREFplayer['GCA_Def'] = pd.to_numeric(FBREFplayer['GCA_Def'], errors='coerce') 

    # Close the driver after scraping
    driver.quit()
    
    return FBREFplayer

### Scraping phase

In [6]:
playerscraper(pl)
playerscraper(liga)
playerscraper(seriea)
playerscraper(bundesliga)
playerscraper(ligue1)


,Player,Nation,Pos,Squad,Age,Born,90s,SCA,SCA90,SCA_PassLive,...,SCA_Fld,SCA_Def,GCA,GCA90,GCA_PassLive,GCA_PassDead,GCA_TO,GCA_Sh,GCA_Fld,GCA_Def
0,Max Aarons,ENG,DF,Bournemouth,24,2000,1.0,2,2.09,2,...,0,0,0,0.00,0,0,0,0,0,0
1,Joshua Acheampong,ENG,DF,Chelsea,18,2006,1.9,2,1.06,2,...,0,0,0,0.00,0,0,0,0,0,0
2,Tyler Adams,USA,MF,Bournemouth,25,1999,21.8,41,1.88,35,...,1,4,4,0.18,3,0,0,0,0,1
3,Tosin Adarabioyo,ENG,DF,Chelsea,26,1997,15.7,15,0.96,8,...,0,1,1,0.06,0,0,0,1,0,0
4,Simon Adingra,CIV,"FW,MF",Brighton,22,2002,12.2,47,3.86,33,...,0,0,7,0.57,5,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2571,Edon Zhegrova,KVX,"FW,MF",Lille,25,1999,10.9,38,3.50,21,...,2,0,3,0.28,2,0,0,1,0,0
2572,Melvin Zinga,FRA,GK,Angers,22,2002,1.0,0,0.00,0,...,0,0,0,0.00,0,0,0,0,0,0
2573,Luck Zogbé,CIV,DF,Brest,19,2005,7.1,15,2.11,8,...,2,3,1,0.14,1,0,0,0,0,0
2574,Aristide Zossou,CIV,MF,Auxerre,19,2005,0.2,0,0.00,0,...,0,0,0,0.00,0,0,0,0,0,0


### export the DataFrame to a CSV file

In [ ]:
FBREFplayer.to_csv('Player - FBREF GCA.csv', index=False)

# Player Defensive Actions

### Create web pages variables to display player defensive statistics by league

In [ ]:
pl = "https://fbref.com/en/comps/9/defense/Premier-League-Stats"
liga = "http://fbref.com/en/comps/12/defense/La-Liga-Stats"
seriea = "https://fbref.com/en/comps/11/defense/Serie-A-Stats"
bundesliga = "https://fbref.com/en/comps/20/defense/Bundesliga-Stats"
ligue1 = "https://fbref.com/en/comps/13/defense/Ligue-1-Stats"


In [25]:
FBREFplayer2 = pd.DataFrame(columns=[
    'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', '90s',
    'Tackles', 'TacklesWon', 'Def3rd_Tkl', 'Mid3rd_Tkl', 'Att3rd_Tkl',
    'DribblersTackled', 'DribblersChallenged', 'Tkl%_vsDrib', 'ChallengesLost',
    'Blocks', 'ShotsBlocked', 'PassesBlocked',
    'Interceptions', 'Clearances','NBPLAYER + Interceptions','Errors'
])

In [26]:
def playerscraper2(url):
    positions = ['GK', 'DF', 'MF', 'FW', 'FW,DF', 'FW,MF', 'MF,DF', 'MF,FW']
    options = Options()
    options.binary_location = brave_path
    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get(url)

    table = driver.find_element(By.XPATH, '//*[@id="stats_defense"]/tbody')
    rows = table.find_elements(By.TAG_NAME, 'tr')

    for row in rows:
        row_data = row.text.split(' ')
        # skip header rows
        if not row_data or row_data[0] == 'Rk':
            continue
        # Remove the first element (the rank)
        row_data.pop(0)
        # Remove the 'Matches' element if it exists
        if 'Matches' in row_data:
            row_data.remove('Matches')
        # for players with name of one word
        if row_data[3] in positions:
            row_data.pop(1)  # remove duplicate nationality
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(4) + ' ' + row_data.pop(4)
                row_data.insert(4, team)
        # for players with name of more than one word
        elif row_data[4] in positions:
            row_data.pop(2)
            name = row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        # for players with name of 3 parts
        elif len(row_data) > 5 and row_data[5] in positions:
            row_data.pop(3)
            name = row_data.pop(0) + ' ' + row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        else:
            continue
        # Check if the row has the expected number of elements
        if len(row_data) == 23:
            FBREFplayer2.loc[len(FBREFplayer2)] = row_data

    # Convert columns to appropriate data types
    FBREFplayer2['90s'] = pd.to_numeric(FBREFplayer2['90s'], errors='coerce')
    FBREFplayer2['Tackles'] = pd.to_numeric(FBREFplayer2['Tackles'], errors='coerce')
    FBREFplayer2['TacklesWon'] = pd.to_numeric(FBREFplayer2['TacklesWon'], errors='coerce')
    FBREFplayer2['Def3rd_Tkl'] = pd.to_numeric(FBREFplayer2['Def3rd_Tkl'], errors='coerce')
    FBREFplayer2['Mid3rd_Tkl'] = pd.to_numeric(FBREFplayer2['Mid3rd_Tkl'], errors='coerce')
    FBREFplayer2['Att3rd_Tkl'] = pd.to_numeric(FBREFplayer2['Att3rd_Tkl'], errors='coerce')
    FBREFplayer2['DribblersTackled'] = pd.to_numeric(FBREFplayer2['DribblersTackled'], errors='coerce')
    FBREFplayer2['DribblersChallenged'] = pd.to_numeric(FBREFplayer2['DribblersChallenged'], errors='coerce')
    FBREFplayer2['Tkl%_vsDrib'] = pd.to_numeric(FBREFplayer2['Tkl%_vsDrib'], errors='coerce')
    FBREFplayer2['ChallengesLost'] = pd.to_numeric(FBREFplayer2['ChallengesLost'], errors='coerce')
    FBREFplayer2['Blocks'] = pd.to_numeric(FBREFplayer2['Blocks'], errors='coerce')
    FBREFplayer2['ShotsBlocked'] = pd.to_numeric(FBREFplayer2['ShotsBlocked'], errors='coerce')
    FBREFplayer2['PassesBlocked'] = pd.to_numeric(FBREFplayer2['PassesBlocked'], errors='coerce')
    FBREFplayer2['Interceptions'] = pd.to_numeric(FBREFplayer2['Interceptions'], errors='coerce')
    FBREFplayer2['Clearances'] = pd.to_numeric(FBREFplayer2['Clearances'], errors='coerce')
    FBREFplayer2['NBPLAYER + Interceptions'] = pd.to_numeric(FBREFplayer2['NBPLAYER + Interceptions'], errors='coerce')
    FBREFplayer2['Errors'] = pd.to_numeric(FBREFplayer2['Errors'], errors='coerce')

    driver.quit()
    return FBREFplayer2

In [ ]:
playerscraper2(pl)
playerscraper2(liga)
playerscraper2(seriea)
playerscraper2(bundesliga)
playerscraper2(ligue1)

,Player,Nation,Pos,Squad,Age,Born,90s,Tackles,TacklesWon,Def3rd_Tkl,...,DribblersChallenged,Tkl%_vsDrib,ChallengesLost,Blocks,ShotsBlocked,PassesBlocked,Interceptions,Clearances,NBPLAYER + Interceptions,Errors
0,Max Aarons,ENG,DF,Bournemouth,24,2000,1.0,2,2,1,...,1,100.0,0,3,1,2,1,3,0,0
1,Joshua Acheampong,ENG,DF,Chelsea,18,2006,1.9,2,1,2,...,2,100.0,0,1,0,1,1,3,2,0
2,Tyler Adams,USA,MF,Bournemouth,25,1999,21.8,83,50,26,...,72,54.2,33,33,10,23,32,115,42,0
3,Tosin Adarabioyo,ENG,DF,Chelsea,26,1997,15.7,17,13,11,...,12,66.7,4,9,7,2,11,28,80,0
4,Simon Adingra,CIV,"FW,MF",Brighton,22,2002,12.2,23,14,10,...,26,38.5,16,12,0,12,8,31,6,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2809,Denis Zakaria,SUI,MF,Monaco,27,1996,23.8,41,26,15,...,33,72.7,9,31,8,23,34,75,39,1
2810,Anass Zaroury,MAR,"FW,MF",Lens,23,2000,15.6,23,17,6,...,26,34.6,17,9,0,9,6,29,15,0
2811,Nathan Zeze,FRA,DF,Nantes,19,2005,16.8,38,22,22,...,33,69.7,10,22,9,13,23,61,120,1
2812,Edon Zhegrova,KVX,"FW,MF",Lille,25,1999,10.9,5,2,3,...,7,28.6,5,8,0,8,4,9,1,0


In [32]:
FBREFplayer2.to_csv('C:\Projects\Predicting Football Player Values\Data\Player - FBREF Defense.csv', index=False)

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\azedd\AppData\Local\Temp\ipykernel_6504\306175494.py:1: SyntaxWarning: invalid escape sequence '\P'
  FBREFplayer2.to_csv('C:\Projects\Predicting Football Player Values\Data\Player - FBREF Defense.csv', index=False)


# Player Miscellaneous Stats

### Let's explore Player Miscellaneous Stats and scrape these characteristics using the same approach as before.

In [6]:
pl = "https://fbref.com/en/comps/9/misc/Premier-League-Stats"
liga = "http://fbref.com/en/comps/12/misc/La-Liga-Stats"
seriea = "https://fbref.com/en/comps/11/misc/Serie-A-Stats"
bundesliga = "https://fbref.com/en/comps/20/misc/Bundesliga-Stats"
ligue1 = "https://fbref.com/en/comps/13/misc/Ligue-1-Stats"

In [15]:
FBREFplayer3 = pd.DataFrame(columns=[
    'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', '90s',
	'CrdY',	'CrdR','2CrdY','Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'PKwon', 'PKcon', 'OG', 'Recov', 'Won','Lost','Won%'
])

In [11]:
def playerscraper3(url):
    positions = ['GK', 'DF', 'MF', 'FW', 'FW,DF', 'FW,MF', 'MF,DF', 'MF,FW']
    options = Options()
    options.binary_location = brave_path
    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get(url)

    table = driver.find_element(By.XPATH, '//*[@id="stats_misc"]/tbody')
    rows = table.find_elements(By.TAG_NAME, 'tr')

    for row in rows:
        row_data = row.text.split(' ')
        # skip header rows
        if not row_data or row_data[0] == 'Rk':
            continue
        # Remove the first element (the rank)
        row_data.pop(0)
        # Remove the 'Matches' element if it exists
        if 'Matches' in row_data:
            row_data.remove('Matches')
        # for players with name of one word
        if row_data[3] in positions:
            row_data.pop(1)  # remove duplicate nationality
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(4) + ' ' + row_data.pop(4)
                row_data.insert(4, team)
        # for players with name of more than one word
        elif row_data[4] in positions:
            row_data.pop(2)
            name = row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        # for players with name of 3 parts
        elif len(row_data) > 5 and row_data[5] in positions:
            row_data.pop(3)
            name = row_data.pop(0) + ' ' + row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        else:
            continue
        # Check if the row has the expected number of elements
        if len(row_data) == len(FBREFplayer3.columns):
            FBREFplayer3.loc[len(FBREFplayer3)] = row_data

    # Convert columns to appropriate data types
    FBREFplayer3['90s'] = pd.to_numeric(FBREFplayer3['90s'], errors='coerce')
    FBREFplayer3['CrdY'] = pd.to_numeric(FBREFplayer3['CrdY'], errors='coerce')
    FBREFplayer3['CrdR'] = pd.to_numeric(FBREFplayer3['CrdR'], errors='coerce')
    FBREFplayer3['2CrdY'] = pd.to_numeric(FBREFplayer3['2CrdY'], errors='coerce')
    FBREFplayer3['Fls'] = pd.to_numeric(FBREFplayer3['Fls'], errors='coerce')
    FBREFplayer3['Fld'] = pd.to_numeric(FBREFplayer3['Fld'], errors='coerce')
    FBREFplayer3['Off'] = pd.to_numeric(FBREFplayer3['Off'], errors='coerce')
    FBREFplayer3['Crs'] = pd.to_numeric(FBREFplayer3['Crs'], errors='coerce')
    FBREFplayer3['Int'] = pd.to_numeric(FBREFplayer3['Int'], errors='coerce')
    FBREFplayer3['TklW'] = pd.to_numeric(FBREFplayer3['TklW'], errors='coerce')
    FBREFplayer3['PKwon'] = pd.to_numeric(FBREFplayer3['PKwon'], errors='coerce')
    FBREFplayer3['PKcon'] = pd.to_numeric(FBREFplayer3['PKcon'], errors='coerce')
    FBREFplayer3['OG'] = pd.to_numeric(FBREFplayer3['OG'], errors='coerce')
    FBREFplayer3['Recov'] = pd.to_numeric(FBREFplayer3['Recov'], errors='coerce')
    FBREFplayer3['Won'] = pd.to_numeric(FBREFplayer3['Won'], errors='coerce')
    FBREFplayer3['Lost'] = pd.to_numeric(FBREFplayer3['Lost'], errors='coerce')
    FBREFplayer3['Won%'] = pd.to_numeric(FBREFplayer3['Won%'], errors='coerce')

    driver.quit()
    return FBREFplayer3

In [16]:
playerscraper3(pl)
playerscraper3(liga)
playerscraper3(seriea)
playerscraper3(bundesliga)
playerscraper3(ligue1)


,Player,Nation,Pos,Squad,Age,Born,90s,CrdY,CrdR,2CrdY,...,Crs,Int,TklW,PKwon,PKcon,OG,Recov,Won,Lost,Won%
0,Joshua Acheampong,ENG,DF,Chelsea,18,2006,1.9,1,0,0,...,0,1,1,0,0,0,7,1,6,14.3
1,Tyler Adams,USA,MF,Bournemouth,25,1999,21.8,7,0,0,...,3,32,50,0,1,0,114,31,18,63.3
2,Tosin Adarabioyo,ENG,DF,Chelsea,26,1997,15.7,4,0,0,...,0,11,13,0,0,0,41,42,28,60.0
3,Simon Adingra,CIV,"FW,MF",Brighton,22,2002,12.2,0,0,0,...,41,8,14,0,0,0,47,7,4,63.6
4,Emmanuel Agbadou,CIV,DF,Wolves,27,1997,15.7,3,0,0,...,2,12,19,0,0,0,83,26,20,56.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2452,Denis Zakaria,SUI,MF,Monaco,27,1996,23.8,5,0,0,...,7,34,26,0,2,0,165,18,12,60.0
2453,Anass Zaroury,MAR,"FW,MF",Lens,23,2000,15.6,3,1,1,...,125,6,17,1,0,0,59,2,5,28.6
2454,Nathan Zeze,FRA,DF,Nantes,19,2005,16.8,6,0,0,...,1,23,22,0,0,0,63,28,25,52.8
2455,Edon Zhegrova,KVX,"FW,MF",Lille,25,1999,10.9,1,0,0,...,58,4,2,0,0,0,42,3,2,60.0


In [17]:
# Save the DataFrame to a CSV file
FBREFplayer3.to_csv('Player - FBREF Misc.csv', index=False)